# move files

In [1]:
1

1

In [2]:
import os
import sys
import shutil
import pickle
from pathlib import Path

# Optional: we use pandas only to read the pickled DataFrame safely
try:
    import pandas as pd  # type: ignore
except Exception:
    pd = None

DATASET_DIR = Path("/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data").resolve()
DEST_DIR = Path("/home/icb/kemal.inecik/lustre_workspace/temp_move").resolve()

# File types we consider as "files to copy" when scanning
ALLOWED_EXTS = {".pkl", ".h5ad", ".csv", ".zarr", ".npz"}

def add_if_file(pathlike, bucket: set[Path]):
    """Normalize and add a candidate file if it exists and has an allowed extension."""
    p = Path(os.path.expanduser(str(pathlike)))
    if not p.is_absolute():
        p = DATASET_DIR / p
    try:
        p = p.resolve()
    except Exception:
        pass
    if p.suffix in ALLOWED_EXTS and p.exists() and p.is_file():
        bucket.add(p)

def walk_collect(obj, bucket: set[Path], max_items=2_000_000):
    """Recursively scrape path-like strings/Paths out of nested containers and pandas objects."""
    # Safety valve to avoid accidental huge traversals
    if len(bucket) > max_items:
        return

    # Primitives / direct path-likes
    if isinstance(obj, (str, os.PathLike, Path)):
        add_if_file(obj, bucket)
        return

    # Containers
    if isinstance(obj, dict):
        for k, v in obj.items():
            walk_collect(k, bucket)
            walk_collect(v, bucket)
        return
    if isinstance(obj, (list, tuple, set)):
        for x in obj:
            walk_collect(x, bucket)
        return

    # Pandas objects (handled lazily, only if pandas is available)
    if pd is not None:
        import pandas as _pd  # noqa
        if isinstance(obj, _pd.Series):
            for x in obj.array:
                walk_collect(x, bucket)
            return
        if isinstance(obj, _pd.DataFrame):
            # Only object-dtype values are likely to contain paths
            for col in obj.columns:
                s = obj[col]
                if s.dtype == "object":
                    for x in s.array:
                        walk_collect(x, bucket)
            return

def main():
    DEST_DIR.mkdir(parents=True, exist_ok=True)

    candidates: set[Path] = set()

    # --- Explicit paths from your snippet ---

    # Pickle bundle referenced by load_pickle_bundle
    pickle_bundle_path = DATASET_DIR / "_experiment_development_atlas_helper_pickle_bundle.pkl"
    candidates.add(pickle_bundle_path)

    # result_df pickle
    candidates.add(DATASET_DIR / "_experiment_development_atlas_result_df_dataframe.pkl")

    # to_pickle outputs
    candidates.add(DATASET_DIR / "_experiment_development_atlas_res_lvl2_dataframe.pkl")
    candidates.add(DATASET_DIR / "_experiment_development_atlas_res_lvl3_dataframe.pkl")
    candidates.add(DATASET_DIR / "_experiment_development_atlas_sctram_summary_dataframe.pkl")
    candidates.add(DATASET_DIR / "_experiment_development_atlas_scib_summary_dataframe.pkl")

    # result_df_main outputs
    candidates.add(DATASET_DIR / "df_boots_fig2_calculations.pkl")
    candidates.add(DATASET_DIR / "df_boots_fig2_calculations_for_individuals.pkl")

    # Explicit .h5ad lines
    explicit_h5ad = [
        "anndata_hdca_input_pns_neuro_litc_2_scpoli.h5ad",
        "anndata_hdca_input_pns_neuro_litc_2_scvi.h5ad",
        "anndata_hdca_input_germline_litc_2_scpoli.h5ad",
        "anndata_hdca_input_germline_litc_2_scvi.h5ad",
        "anndata_hdca_input_Haem_ref_litc_2_scpoli_nodes_ery.h5ad",
        "anndata_hdca_input_Haem_ref_litc_2_scvi_nodes_ery.h5ad",
        "anndata_hdca_input_Haem_ref_litc_2_scpoli_nodes_b.h5ad",
        "anndata_hdca_input_Haem_ref_litc_2_scvi_nodes_b.h5ad",
        # (the 'ery' pair appears twice in your snippet; we include it once)
    ]
    for name in explicit_h5ad:
        candidates.add((DATASET_DIR / name).resolve())

    # Trajectory focus pkl files
    for trajectory_focus in ["Haem_ref", "germline", "pns_neuro"]:
        candidates.add((DATASET_DIR / f"adata_hdca_input_{trajectory_focus}_litc_2.pkl").resolve())
        candidates.add((DATASET_DIR / f"adata_hdca_input_{trajectory_focus}_litc_1.pkl").resolve())

    # Comparison h5ad files
    candidates.add((DATASET_DIR / "anndata_hdca_input_germline_litc_comparison_atlas.h5ad").resolve())
    candidates.add((DATASET_DIR / "anndata_hdca_input_germline_litc_comparison_comp.h5ad").resolve())

    # --- Try to discover any additional file paths inside the pickle bundle/result_df ---

    if pickle_bundle_path.exists():
        try:
            with pickle_bundle_path.open("rb") as f:
                bundle_obj = pickle.load(f)
            walk_collect(bundle_obj, candidates)
        except Exception as e:
            print(f"[warn] Could not parse bundle {pickle_bundle_path.name}: {e}", file=sys.stderr)

    result_df_pickle = DATASET_DIR / "_experiment_development_atlas_result_df_dataframe.pkl"
    if pd is not None and result_df_pickle.exists():
        try:
            df_obj = pd.read_pickle(result_df_pickle)
            walk_collect(df_obj, candidates)
        except Exception as e:
            print(f"[warn] Could not parse {result_df_pickle.name}: {e}", file=sys.stderr)

    # --- Filter to allowed extensions and existing files ---
    final_sources: list[Path] = []
    for p in sorted({p for p in candidates if p.suffix in ALLOWED_EXTS}):
        if p.exists() and p.is_file():
            final_sources.append(p)
        else:
            # We keep silent here; we'll summarize at the end.
            pass

    if not final_sources:
        print("No matching files were found to copy. "
              "Check that DATASET_DIR is correct and files exist.", file=sys.stderr)
        sys.exit(1)

    # --- Copy, avoiding overwrites by adding a suffix if needed ---
    copied = []
    missing = []
    for src in final_sources:
        if not src.exists():
            missing.append(src)
            continue
        dst = DEST_DIR / src.name
        if dst.exists():
            # If the exact same file is attempting to copy twice, skip duplicating.
            try:
                if src.resolve() == dst.resolve():
                    continue
            except Exception:
                pass
            i = 1
            while True:
                candidate = DEST_DIR / f"{src.stem}__{i}{src.suffix}"
                if not candidate.exists():
                    dst = candidate
                    break
                i += 1
        shutil.copy2(src, dst)
        copied.append((src, dst))

    # --- Summary ---
    print(f"Destination: {DEST_DIR}")
    print(f"Copied {len(copied)} files.")
    if missing:
        print(f"Missing (not found) {len(missing)} files:")
        for m in missing:
            print(f"  - {m}")

    # Exit non-zero if anything referenced didn't exist (so you don't silently miss files)
    if missing:
        sys.exit(2)

if __name__ == "__main__":
    main()


Destination: /ictstr01/groups/ml01/workspace/kemal.inecik/temp_move
Copied 46 files.


In [3]:
import os

total_size_bytes = 0
for f in DEST_DIR.iterdir():
    if f.is_file():
        total_size_bytes += f.stat().st_size

total_size_gb = total_size_bytes / (1024 ** 3)
total_size_mb = total_size_bytes / (1024 ** 2)

print(f"Copied files total size: {total_size_bytes:,} bytes")
print(f"≈ {total_size_mb:.2f} MB")
print(f"≈ {total_size_gb:.2f} GB")


Copied files total size: 62,274,771,584 bytes
≈ 59389.85 MB
≈ 58.00 GB


In [10]:
import pandas as pd
from pathlib import Path

# Ensure DEST_DIR exists from earlier; otherwise set it here:
# DEST_DIR = Path("/home/icb/kemal.inecik/lustre_workspace/temp_move")

# Collect files (non-recursive; switch to rglob('*') for recursive)
entries = []
for p in sorted(DEST_DIR.iterdir()):
    if p.is_file():
        size_bytes = p.stat().st_size
        entries.append({
            "File": p.name,                      # or str(p) for full path
            "Size (MB)": round(size_bytes / (1024**2), 2),
            "Size (GB)": round(size_bytes / (1024**3), 4),
            "Bytes": size_bytes
        })

df = pd.DataFrame(entries).sort_values("Bytes", ascending=False).reset_index(drop=True)

# 1) Show full filenames in the DataFrame (no truncation)
with pd.option_context(
    "display.max_colwidth", None,   # don't truncate long strings
    "display.width", None,          # auto-detect width
    "display.max_rows", None        # show all rows if you want; change if huge
):
    display(df[["File", "Size (MB)", "Size (GB)", "Bytes"]])


,File,Size (MB),Size (GB),Bytes
0,unification_union_20240330_hvg-intersection_integration.h5ad,38843.49,37.9331,40730355118
1,unification_union_20240330_hvg_integration.h5ad,10046.97,9.8115,10535011118
2,unification_union_20240330_hvg-intersection_integration_Braun_scvi.h5ad,2172.63,2.1217,2278163605
3,unification_union_20240330_hvg-intersection_integration_X_scanorama.h5ad,1514.54,1.4790,1588114200
4,unification_union_20240330_hvg-intersection_integration_Suo_scvi.h5ad,1108.53,1.0826,1162380992
5,pca_unification_union_20240330_hvg_integration.h5ad,826.13,0.8068,866261200
6,harmony_concatenate_unification_union_20240330_hvg-intersection_integration.h5ad,826.13,0.8068,866261200
7,20240614_182544898191_gpusrv74.scidom.de_567700.h5ad,729.20,0.7121,764625406
8,20240613_152640100145_gpusrv44.scidom.de_2367215.h5ad,729.20,0.7121,764625406
9,20240615_165112920733_gpusrv29.scidom.de_1630777.h5ad,729.20,0.7121,764625406
